## Local Setup

Run this notebook from your local rc-foundry checkout. Launch Jupyter with `PYTHONNOUSERSITE=1 pixi run jupyter lab` (or notebook) so it stays inside the pixi environment. Update the checkpoint paths in the next cell to point at your locally stored weights—no downloads are performed here.


### Optional: Install from GitHub (Colab)
If you're running in Google Colab, run this to install rc-foundry directly from the fork at https://github.com/magnusbauer/foundry.

In [ ]:
!git clone https://github.com/magnusbauer/ptm_foundry.git foundry

Cloning into 'foundry'...
remote: Enumerating objects: 10384, done.
remote: Counting objects: 100% (738/738), done.
remote: Compressing objects: 100% (219/219), done.
remote: Total 10384 (delta 586), reused 523 (delta 518), pack-reused 9646 (from 2)
Receiving objects: 100% (10384/10384), 81.31 MiB | 17.11 MiB/s, done.
Resolving deltas: 100% (6736/6736), done.


In [ ]:
# !rm -r foundry/

In [ ]:
%%time

import os, sys, subprocess
from pathlib import Path

os.environ["CCD_MIRROR_PATH"] = ""
os.environ["PDB_MIRROR_PATH"] = ""

CKPT_DIR = Path("/root/.foundry/checkpoints/")
CKPT_DIR.mkdir(parents=True, exist_ok=True)
READY_FLAG = Path("FOUNDRY_READY")
LOG = Path("foundry_setup.log")

CHECKPOINTS = {
    "rfd3": {
        "url": "https://files.ipd.uw.edu/pub/rfd3/rfd3_foundry_2025_12_01_remapped.ckpt",
        "filename": "rfd3_latest.ckpt",
    },
    "ligandmpnn": {
        "url": "https://files.ipd.uw.edu/pub/ligandmpnn/ligandmpnn_v_32_010_25.pt",
        "filename": "ligandmpnn_v_32_010_25.pt",
    },
    "rf3": {
        "url": "https://files.ipd.uw.edu/pub/rf3/rf3_foundry_01_24_latest_remapped.ckpt",
        "filename": "rf3_foundry_01_24_latest_remapped.ckpt",
    },
}

# Always remove torchvision first
subprocess.check_call([sys.executable, "-m", "pip", "uninstall", "-y", "torchvision"])

# Start rc-foundry install in the background (if not done)
pip_proc = None
if not READY_FLAG.exists():
    print("Installing rc-foundry (background)...")
    pip_proc = subprocess.Popen(
        [sys.executable, "-m", "pip", "install", "-q", "rc-foundry[all]", "plotly>=5,<7"],
        stdout=LOG.open("ab"),
        stderr=subprocess.STDOUT,
    )
else:
    print("rc-foundry already installed.")

# Start checkpoint downloads in parallel with correct filenames
dl_procs = []
for name, info in CHECKPOINTS.items():
    dest = CKPT_DIR / info["filename"]
    if dest.exists():
        continue
    print(f"Downloading {name} -> {dest} (background)...")
    dl_procs.append(
        subprocess.Popen(
            ["curl", "-L", "-o", str(dest), info["url"]],
            stdout=LOG.open("ab"),
            stderr=subprocess.STDOUT,
        )
    )

# Wait when you actually need everything ready
if pip_proc:
    rc = pip_proc.wait()
    if rc == 0:
        READY_FLAG.touch()
    else:
        print(f"pip install failed with code {rc}")
for p in dl_procs:
    p.wait()

print("Setup steps finished (see foundry_setup.log).")


Installing rc-foundry (background)...
Setup steps finished (see foundry_setup.log).
CPU times: user 20.8 ms, sys: 679 µs, total: 21.5 ms
Wall time: 7min 44s


In [ ]:
import sys, os

repo = "/content/foundry"  # your clone
sys.path[:0] = [
    # os.path.join(repo, "src"),                # foundry, foundry_cli
    os.path.join(repo, "models", "rfd3", "src"),  # rfd3 package
    # add these too if you want to override rf3/mpnn:
    # os.path.join(repo, "models", "rf3", "src"),
    # os.path.join(repo, "models", "mpnn", "src"),
]


# Example: End-To-End *De Novo* Protein Design Pipeline

## Overview

This notebook demonstrates an end-to-end protein design workflow using three deep learning networks from the Institute for Protein Design:

| Step | Model | Purpose |
|------|-------|---------|
| 1. **Generation** | RFD3 | Generate novel proteins via diffusion |
| 2. **Sequence Design** | MPNN | Design amino acid sequences for the generated backbone |
| 3. **Structure Validation via Refolding** | RF3 | Predict the structure from designed sequence to validate designability |

All models are unified through [AtomWorks](https://github.com/RosettaCommons/atomworks) (for both inference and training), relying on Biotite `AtomArray` objects.

### Pipeline Flow
```
RFD3 (backbone) → MPNN (sequence) → RF3 (validation) → RMSD comparison
```

---

In [ ]:
import warnings
warnings.filterwarnings('ignore', module='atomworks')

# Shared utilities for visualization (from AtomWorks)
from atomworks.io.utils.visualize import view

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "user": "Session.username",


## Section 1: All-Atom Generation with RFD3

RFdiffusion3 (RFD3) generates *de novo* all-atom proteins that meet specific conditioning requirements.

**Parameters Used** *(many more are available for more complex protein design tasks)*:
- `length`: Target protein length in residues
- `diffusion_batch_size`: Number of structures to generate per batch
- `n_batches`: Number of batches to run

**Outputs:** Dictionary of `RFD3Output` objects.

In [ ]:
from lightning.fabric import seed_everything
from rfd3.engine import RFD3InferenceConfig, RFD3InferenceEngine

# Set seed for reproducibility
seed_everything(1)

# Configure RFD3 inference
config = RFD3InferenceConfig(
    specification={
        'length': 80,  # Generate 80-residue proteins
    },
    diffusion_batch_size=2,  # Generate 2 structures per batch
)

# Initialize engine and run generation
model = RFD3InferenceEngine(**config)
outputs = model.run(
    inputs=None,      # None for unconditional generation
    out_dir=None,     # None to return in memory (no file output)
    n_batches=1,      # Generate 1 batch
)

<frozen importlib._bootstrap>:488: DeprecationWarning: builtin type SwigPyPacked has no __module__ attribute
<frozen importlib._bootstrap>:488: DeprecationWarning: builtin type SwigPyObject has no __module__ attribute
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "user": "Session.username",
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "user": "Session.username",
INFO:foundry:cuEquivariance is available and will be used.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow

In [7]:
# View generated example IDs (one key per generated structure)
outputs.keys()

dict_keys(['_0'])

In [8]:
# Inspect RFD3 outputs and extract the generated structures
for idx, data in outputs.items():
    print(f"Batch {idx}: {len(data)} structure(s)")
    # print(f"  Output type: {type(data[0]).__name__}")
    # print(f"  AtomArray: {data[0].atom_array}")

# Extract the first generated structure for downstream use
first_key = next(iter(outputs.keys()))
atom_array = outputs[first_key][0].atom_array

# Visualize the generated structure
view(atom_array)

Batch _0: 2 structure(s)


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

### Optional: Run benchmark 1a81 (CD3e)

Loads the example design specification from `benchmarks/1a81.json` and uses the resulting backbone for downstream steps.


In [ ]:
from pathlib import Path

benchmark_json = Path("/content/foundry/examples/1a81.json")
if not benchmark_json.is_file():
    raise FileNotFoundError(f"Benchmark file not found: {benchmark_json}")

print(f"Running RFD3 on {benchmark_json}")

# Build a fresh engine (avoid carrying over overrides like a fixed length)
benchmark_config = RFD3InferenceConfig(
    diffusion_batch_size=1,
)
benchmark_model = RFD3InferenceEngine(**benchmark_config)

outputs = benchmark_model.run(
    inputs=str(benchmark_json),
    out_dir=None,
    n_batches=1,
)


INFO:rfd3.engine:[rank: 0] Prevalidating design specification for example: 1a81_cd3e


Running RFD3 on /content/foundry/examples/1a81.json


INFO: Using bfloat16 Automatic Mixed Precision (AMP)
INFO:lightning.pytorch.utilities.rank_zero:Using bfloat16 Automatic Mixed Precision (AMP)
INFO:rfd3.engine:[rank: 0] Finished inference batch in 15.50 seconds.


In [ ]:
# View generated example IDs (one key per generated structure)
outputs.keys()

dict_keys(['1a81_cd3e_0'])

In [ ]:
# Inspect RFD3 outputs and extract the generated structures
for idx, data in outputs.items():
    print(f"Batch {idx}: {len(data)} structure(s)")
    # print(f"  Output type: {type(data[0]).__name__}")
    # print(f"  AtomArray: {data[0].atom_array}")

# Extract the first generated structure for downstream use
first_key = next(iter(outputs.keys()))
atom_array = outputs[first_key][0].atom_array

# Visualize the generated structure
view(atom_array)

Batch 1a81_cd3e_0: 1 structure(s)


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [ ]:
atom_array[atom_array.res_name=='PTR']

AtomArray([
	Atom(np.array([-2.183158  ,  0.19070004, 10.979193  ], dtype=float32), chain_id="B", res_id=3, ins_code="", res_name="PTR", hetero=True, atom_name="N", element="N", is_C_terminus=False, label_entity_id=2, src_component="A3", chain_iid="B_1", is_covalent_modification=False, is_polymer=True, within_poly_res_idx=2, is_motif_atom_with_fixed_seq=True, is_motif_atom_with_fixed_coord=False, is_motif_atom_unindexed_motif_breakpoint=False, is_dna=False, is_N_terminus=False, is_motif_atom_unindexed=False, pn_unit_iid="B_1", atom_id=65, atomic_number=7, ref_plddt=1, charge=0.0, is_ligand=False, is_non_loopy_atom_level=0, atomize=True, pn_unit_entity=1, is_non_loopy=0, molecule_id=1, token_id=12, is_rna=False, b_factor=3.131347209206315e-31, orig_res_id=3, chain_entity=1, molecule_entity=1, occupancy=1.0, is_residue=False, chain_type=6, pn_unit_id="B", is_protein=True, within_chain_res_idx=2, gt_atom_name="N", is_backbone=True, is_sidechain=False, coord_to_be_noised=[0. 0. 0.]),
	Atom

---

## Section 2: Sequence Design with MPNN

Protein and Ligand MPNN (Message Passing Neural Network) designs amino acid sequences that will fold into a target backbone structure.

**Model Options:**
- `protein_mpnn`: Original ProteinMPNN for protein-only design
- `ligand_mpnn`: Extended model supporting ligand-aware design

**Key Parameters:**
- `batch_size`: Number of sequences to generate per structure
- `remove_waters`: Whether to exclude water molecules from context

In [ ]:
from mpnn.inference_engines.mpnn import MPNNInferenceEngine

# Configure MPNN inference engine
# See mpnn.utils.inference.MPNN_GLOBAL_INFERENCE_DEFAULTS for all options
engine_config = {
    "model_type": "ligand_mpnn",  # or "protein_mpnn" for vanilla ProteinMPNN
    "is_legacy_weights": True,    # Required for now for ligand_mpnn and protein_mpnn
    "out_directory": None,        # Return results in memory
    "write_structures": False,
    "write_fasta": False,
}

# Configure per-input inference options
# See mpnn.utils.inference.MPNN_PER_INPUT_INFERENCE_DEFAULTS for all options
input_configs = [
    {
        "batch_size": 10,         # Generate 10 sequences per structure
        "remove_waters": True,
    }
]

# Run sequence design on the RFD3-generated backbone
model = MPNNInferenceEngine(**engine_config)
mpnn_outputs = model.run(input_dicts=input_configs, atom_arrays=[atom_array])


In [ ]:
from biotite.structure import get_residue_starts
from atomworks.constants import DICT_THREE_TO_ONE, UNKNOWN_AA

# Extract and display the designed sequences
print(f"Generated {len(mpnn_outputs)} designed sequences:\n")

three_to_one = {**DICT_THREE_TO_ONE, 'PTR': 'Y', 'M3L': 'K'}  # extend AtomWorks map for phosphotyrosine
unknown = DICT_THREE_TO_ONE[UNKNOWN_AA]

for i, item in enumerate(mpnn_outputs):
    res_starts = get_residue_starts(item.atom_array)
    seq_1letter = ''
    for res_name in item.atom_array.res_name[res_starts]:
        res3 = res_name.upper()
        seq_1letter += three_to_one.get(res3, unknown)
    print(f"Sequence {i+1}: {seq_1letter}")


Generated 10 designed sequences:

Sequence 1: MSLVDQLEAALLALIDTRRAAAAVAALREAEKVVAAVAEAARAARADELAALFERVIELLRELAEAVAAERWDEARELAARIAELLREARRLAAASPTLSAIVDTALEQLTAIFTAAAAAGHPDPDYLPVVRV
Sequence 2: MSLVDRLLAALLALIDTRDAAAALAAIEEAIKIVRAMEEAARAAKAEEAAALFAEVVRLLRELAAAVAAEEWARARALARRIAALLREARRLAAASPTLAAIVDTALEEMTAIFEAAARAGTPNPEYLPIVRS
Sequence 3: MSLVDRLLAALEALIDSRRAAAAQAAIAQAQGVVAAVAAAARAARMDEAAALFERVLELLAQLSEAVAAERWAEARELAREIAEVLREARRLAAASATLSAIVETALEQMTAIFEAAAAAGTPDPTYLPVVRV
Sequence 4: MSLVDRLEQALLALIDTRRAAAAVAAIAEARKIVAAMAAAAREAEADELAALFEEVLRLLEELAAAVAARRWEEARALARRIAELLRRARELAAASPTLSAIVETALEQMTAIFEAAAAAGTPDPEYLPVVRV
Sequence 5: MSLVDQLLAALEALIDTRNAAAAIAALDQAVGVVRAVAEAARLAKADEAAAIFERVVELLGELKRAVAAENWDEARALAREIAALLREARRLAASSPTLSAIVEEALRQMTAIFTAAAAKGTPDPTYLPVVRV
Sequence 6: MSLVDQLEAALLALIDTRDAAAAVAALEQAIEVVRAMEAAARAAKADELAAIFARVVELLRELSAAVAAERWEEARRLAAEIARLLREGLELAKSSKTLSKIVEEALRQLTEIFQAAAAAGTPDPTYLPIVRV
Sequence 7: MSLVDQLLAALEALIDTRNAAAAVAAIEQALKVIADMAELARQAKAEEMAALFERVQELLKQLAAAVAAKNWDEARAL

---

## Section 3: Structure Prediction with RF3

RF3 (RoseTTAFold 3) predicts protein structures from sequences. By re-folding the MPNN-designed sequence, we can validate whether the design is likely to adopt the intended backbone structure.

**Outputs:** `RF3Output` objects containing:
- `atom_array`: Predicted structure as Biotite AtomArray
- `summary_confidences`: Overall confidence metrics (pLDDT, PAE, pTM, etc.)
- `confidences`: Per-atom/residue confidence scores

**Confidence Metrics:**
| Metric | Description |
|--------|-------------|
| pLDDT | Per-residue confidence (0-1, higher is better) |
| PAE | Predicted Aligned Error (lower is better) |
| pTM | Predicted TM-score |
| ranking_score | Overall model quality score |

In [ ]:
from rf3.inference_engines.rf3 import RF3InferenceEngine
from rf3.utils.inference import InferenceInput


# Initialize RF3 inference engine
inference_engine = RF3InferenceEngine(ckpt_path='rf3', verbose=False)

# Create input from the MPNN-designed structure (first design)
# This re-folds the sequence to validate it adopts the intended structure
input_structure = InferenceInput.from_atom_array(atom_array, example_id="example_protein")
# rf3_outputs = inference_engine.run(inputs=input_structure)
rf3_outputs = inference_engine.run(
    inputs=input_structure,
    annotate_b_factor_with_plddt=True,
    # out_dir="rf3_out",  # optional: also write CIFs with pLDDT in B-factor
)

# Outputs: dict mapping example_id -> list[RF3Output] (multiple models per input)
print(f"Output keys: {rf3_outputs.keys()}")
print(f"Number of models for 'example_protein': {len(rf3_outputs['example_protein'])}")

INFO:rf3.inference_engines.rf3:[rank: 0] Loading checkpoint from /root/.foundry/checkpoints/rf3_foundry_01_24_latest_remapped.ckpt...
INFO: Using bfloat16 Automatic Mixed Precision (AMP)
INFO:lightning.pytorch.utilities.rank_zero:Using bfloat16 Automatic Mixed Precision (AMP)
INFO:rf3.inference_engines.rf3:[rank: 0] Found 1 structures to predict!
INFO:rf3.inference_engines.rf3:[rank: 0] Predicting structure 1/1: example_protein


Output keys: dict_keys(['example_protein'])
Number of models for 'example_protein': 5


In [ ]:
# Extract the top-ranked prediction
rf3_output = rf3_outputs["example_protein"][0]

# Inspect RF3Output structure
print(f"RF3Output contains:")
print(f"  - atom_array: {len(rf3_output.atom_array)} atoms")
print(f"  - summary_confidences: {list(rf3_output.summary_confidences.keys())}")
print(f"  - confidences: {list(rf3_output.confidences.keys()) if rf3_output.confidences else None}")

# Visualize the predicted structure
view(rf3_output.atom_array)

RF3Output contains:
  - atom_array: 915 atoms
  - summary_confidences: ['chain_ptm', 'chain_pair_pae_min', 'chain_pair_pde_min', 'chain_pair_pae', 'chain_pair_pde', 'overall_plddt', 'overall_pde', 'overall_pae', 'ptm', 'iptm', 'has_clash', 'ranking_score']
  - confidences: ['atom_chain_ids', 'atom_plddts', 'pae', 'token_chain_ids', 'token_res_ids']


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [ ]:
# Summary confidences: overall model quality metrics
summary = rf3_output.summary_confidences

print("=== Summary Confidences ===")
print(f"  Overall pLDDT:    {summary['overall_plddt']:.3f}")
print(f"  Overall PAE:      {summary['overall_pae']:.2f} A")
print(f"  Overall PDE:      {summary['overall_pde']:.3f}")
print(f"  pTM:              {summary['ptm']:.3f}")
print(f"  ipTM:             {summary.get('iptm', 'N/A (single chain)')}")
print(f"  Ranking score:    {summary['ranking_score']:.3f}")
print(f"  Has clash:        {summary['has_clash']}")

=== Summary Confidences ===
  Overall pLDDT:    0.728
  Overall PAE:      12.97 A
  Overall PDE:      4.856
  pTM:              0.534
  ipTM:             0.2305687814950943
  Ranking score:    0.291
  Has clash:        False


In [ ]:
# Detailed per-atom/residue confidences
conf = rf3_output.confidences

print("=== Per-Atom/Residue Confidences ===")
print(f"  atom_plddts:      {len(conf['atom_plddts'])} values (one per atom)")
print(f"  atom_chain_ids:   {len(conf['atom_chain_ids'])} values")
print(f"  token_chain_ids:  {len(conf['token_chain_ids'])} values (one per residue)")
print(f"  token_res_ids:    {len(conf['token_res_ids'])} values")
print(f"  PAE matrix:       {len(conf['pae'])}x{len(conf['pae'][0])}")

# Preview first 10 atom pLDDT scores
import numpy as np
print(f"\nFirst 10 atom pLDDTs: {np.round(conf['atom_plddts'][:10], 2).tolist()}")

=== Per-Atom/Residue Confidences ===
  atom_plddts:      915 values (one per atom)
  atom_chain_ids:   915 values
  token_chain_ids:  148 values (one per residue)
  token_res_ids:    148 values
  PAE matrix:       148x148

First 10 atom pLDDTs: [0.68, 0.69, 0.7, 0.67, 0.67, 0.65, 0.63, 0.58, 0.7, 0.71]


In [ ]:
np.round(conf['atom_plddts'][:10], 2).tolist()

[0.68, 0.69, 0.7, 0.67, 0.67, 0.65, 0.63, 0.58, 0.7, 0.71]

---

## Section 4: Validation and Export

The final step compares the RF3-predicted structure against the original RFD3-generated backbone. A low backbone RMSD indicates the designed sequence is likely to fold into the intended structure (high designability).

In [ ]:
from biotite.structure import rmsd, superimpose
from atomworks.constants import PROTEIN_BACKBONE_ATOM_NAMES
import numpy as np

# Get structures for comparison
aa_generated = atom_array              # Original RFD3 backbone (from Section 1)
aa_refolded = rf3_output.atom_array    # RF3-predicted structure

# Filter to backbone atoms (N, CA, C, O)
bb_generated = aa_generated[np.isin(aa_generated.atom_name, PROTEIN_BACKBONE_ATOM_NAMES)]
bb_refolded = aa_refolded[np.isin(aa_refolded.atom_name, PROTEIN_BACKBONE_ATOM_NAMES)]

# Superimpose structures and calculate RMSD
bb_refolded_fitted, _ = superimpose(bb_generated, bb_refolded)
rmsd_value = rmsd(bb_generated, bb_refolded_fitted)

print(f"Backbone RMSD: {rmsd_value:.2f} A")
print(f"\nInterpretation: {'Excellent' if rmsd_value < 1.0 else 'Good' if rmsd_value < 2.0 else 'Moderate'} designability")

Backbone RMSD: 9.79 A

Interpretation: Moderate designability


In [ ]:
from atomworks.io.utils.io_utils import to_cif_file

# Export structures to CIF format for visualization in PyMOL/ChimeraX
to_cif_file(aa_generated, "generated.cif")
to_cif_file(aa_refolded, "refolded.cif")

print("Exported structures:")
print("  - generated.cif: Original RFD3 backbone")
print("  - refolded.cif:  RF3-predicted structure")

Exported structures:
  - generated.cif: Original RFD3 backbone
  - refolded.cif:  RF3-predicted structure
